# NB10 — VLM V2 real integration test (VALID, T4-safe)

Notebook này **chỉ dùng để test end-to-end VLM V2**, không phải production deployment.

Pipeline:

`ML_Final → frozen scorer + LOO → Recommendation V2 → fetch vài ảnh cần thiết từ Hugging Face → vlm-evidence-v2 → Qwen3-VL → validator → renderer → handoff`

Điểm quan trọng:
- không cần Drive `images/` 142k ảnh;
- không đưa `negative_metadata`, swapped/original GT hay evaluation target vào VLM;
- Qwen chỉ load **một lần** trong runtime;
- trên T4, notebook dùng memory-safe loader + pixel-budget fallback để tránh OOM;
- pixel budget T4 trong NB10 chỉ là **functional-test override**, không thay canonical deploy config.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"
BRANCH = "feat/vlm-recommendation-evidence-v2"
REPO_ROOT = Path("/content/opisoverated")

if not REPO_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_ROOT)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", BRANCH], check=True)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from google.colab import drive
drive.mount("/content/drive")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "requirements-vlm.txt")],
    check=True,
)

HEAD = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True
).strip()
print("Branch:", BRANCH)
print("HEAD  :", HEAD)


## 1. ML_Final trên Drive

Nếu `ML_Final` chỉ nằm ở **Shared with me**, hãy Add shortcut vào My Drive. Không cần folder `images`.


In [ ]:
ARTIFACT_ROOT = Path("/content/drive/MyDrive/ML_Final")
assert ARTIFACT_ROOT.is_dir(), (
    f"Không thấy ML_Final tại {ARTIFACT_ROOT}. "
    "Nếu folder chỉ ở Shared with me, hãy Add shortcut to Drive rồi sửa path này."
)
print("ML_Final:", ARTIFACT_ROOT)


## 2. Fail-fast unit tests

Chạy trước khi load Qwen để bắt schema/prompt/validator bug mà không tốn VRAM.


In [ ]:
patterns = [
    "test_vlm_evidence_v2.py",
    "test_vlm_prompt_v2.py",
    "test_vlm_prompt_v2_anti_anchor.py",
    "test_vlm_validator_v2.py",
    "test_vlm_pipeline_v2.py",
    "test_vlm_explanation.py",
]
for pattern in patterns:
    run = subprocess.run(
        [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", pattern, "-v"],
        cwd=REPO_ROOT,
        text=True,
        capture_output=True,
    )
    print(run.stdout)
    if run.stderr:
        print(run.stderr)
    if run.returncode != 0:
        raise RuntimeError(f"Tests failed: {pattern}")
print("VLM V1 + V2 TESTS: PASS")


## 3. Load frozen Recommendation artifacts — CPU only

Recommendation/scorer ở CPU để dành toàn bộ VRAM cho Qwen. Resolver bên dưới chỉ là NB10 test resolver; ảnh thật sẽ được fetch sau khi Top-3 đã được chọn.


In [ ]:
import torch
from urllib.parse import quote

from src.recommendation.pipeline import RecommendationPipeline
from src.recommendation.directory_artifacts import MLFinalDirectoryBundle

assert torch.cuda.is_available(), "Colab: Runtime → Change runtime type → T4 GPU"

REC_CONFIG_PATH = REPO_ROOT / "configs" / "recommendation_category_aware_v2.json"
rec_config = RecommendationPipeline._load_config(REC_CONFIG_PATH)

bundle = MLFinalDirectoryBundle(ARTIFACT_ROOT)
catalog = bundle.load_embedding_catalog()
metadata = bundle.load_metadata_index()

class CatalogImageAvailabilityResolver:
    """NB10-only: use frozen catalog membership; do not require local image archives."""
    def __init__(self, item_ids):
        self.item_ids = tuple(str(x) for x in item_ids)
        self._item_set = set(self.item_ids)

    def __contains__(self, item_id):
        return str(item_id) in self._item_set

    def image_url(self, item_id, *, base_path="/recommendation/images"):
        item_id = str(item_id)
        if item_id not in self._item_set:
            raise KeyError(item_id)
        return f"{base_path.rstrip('/')}/{quote(item_id, safe='')}"

image_availability = CatalogImageAvailabilityResolver(catalog.item_ids)

rec_pipeline = RecommendationPipeline._build(
    config=rec_config,
    bundle=bundle,
    catalog=catalog,
    metadata=metadata,
    image_resolver=image_availability,
    device="cpu",
)

print("GPU:", torch.cuda.get_device_name(0))
print("Catalog embeddings:", len(catalog))
print("Declared image availability:", len(image_availability.item_ids))
print("Mapping exact:", rec_pipeline.image_validation["mapping_exact"])


## 4. Chọn deterministic một VALID negative có đúng 4 item

Dùng `label` chỉ để chọn một case negative dễ quan sát. Không đọc `negative_metadata` hay synthetic GT. Chọn đúng 4 item để giới hạn visual input thành **4 outfit crops + 3 recommendation images**.


In [ ]:
from src.diagnosis.loo import diagnose_outfit

valid_records = bundle.load_scorer_ready("valid")
candidates = sorted(
    (
        row for row in valid_records
        if int(row.get("label", 1)) == 0 and len(row.get("items", [])) == 4
    ),
    key=lambda row: str(row.get("sample_id", "")),
)
assert candidates, "Không tìm thấy VALID negative có đúng 4 item"
sample = candidates[0]

item_ids = [str(x) for x in sample["items"]]
outfit_embeddings = catalog.get_embeddings(item_ids)
outfit_category_ids = [int(metadata.category_id(item_id)) for item_id in item_ids]

loo_result = diagnose_outfit(
    rec_pipeline.reranker.scorer,
    torch.as_tensor(outfit_embeddings, dtype=torch.float32),
    torch.as_tensor(outfit_category_ids, dtype=torch.long),
    item_ids=item_ids,
)

problem_index = int(loo_result["problematic_item_index"])
recommendation_result = rec_pipeline.recommend(
    outfit_item_ids=item_ids,
    outfit_embeddings=outfit_embeddings,
    outfit_category_ids=outfit_category_ids,
    problematic_index=problem_index,
    loo_result=loo_result,
    query_id=str(sample["sample_id"]),
    source_split="valid",
)

print("Sample:", sample["sample_id"])
print("Outfit:", item_ids)
print("LOO problematic:", problem_index, loo_result["problematic_item_id"])
print("Top-3:", [row.item_id for row in recommendation_result.items])


## 5. Build `vlm-evidence-v2`

Evidence chứa scorer + LOO + authoritative Recommendation Top-3. Có assertion chống evaluation leakage trước khi Qwen nhìn thấy evidence.


In [ ]:
from src.vlm import build_vlm_evidence_v2

coarse_categories = [str(metadata.coarse_category(item_id)) for item_id in item_ids]

evidence = build_vlm_evidence_v2(
    loo_result,
    recommendation_result,
    sample_id=str(sample["sample_id"]),
    item_ids=item_ids,
    coarse_categories=coarse_categories,
)

serialized = json.dumps(evidence, ensure_ascii=False)
for forbidden in (
    "negative_metadata",
    "swapped_item_index",
    "target_swapped_item_index",
    "ground_truth_item_id",
    "original_item_id",
):
    assert forbidden not in serialized, f"Leakage detected: {forbidden}"

print("Evidence schema:", evidence["schema_version"])
print(
    "Evidence Top-3:",
    [(x["rank"], x["item_id"]) for x in evidence["recommendation"]["items"]],
)


## 6. Fetch đúng 7 ảnh cần dùng từ Hugging Face

Không tải toàn bộ Polyvore. Chỉ lấy 4 outfit images và 3 candidate images được chọn.


In [ ]:
from datasets import load_dataset
from PIL import Image

wanted_ids = set(item_ids)
wanted_ids.update(row.item_id for row in recommendation_result.items)

images_by_id = {}
missing = set(wanted_ids)

for split in ("valid", "train", "test"):
    if not missing:
        break
    try:
        source_items = load_dataset(
            "codewaly/polyvore1000",
            "items",
            split=split,
            streaming=True,
        )
    except Exception as error:
        print(f"Skip split {split}: {type(error).__name__}: {error}")
        continue

    print(f"Scanning HF items/{split} for {len(missing)} remaining image(s)...")
    for row in source_items:
        item_id = str(row["item_id"])
        if item_id not in missing:
            continue
        image = row.get("image")
        if not isinstance(image, Image.Image):
            raise TypeError(f"HF image for {item_id} is not PIL.Image")
        images_by_id[item_id] = image.convert("RGB")
        missing.remove(item_id)
        if not missing:
            break

assert not missing, f"Không fetch được ảnh cho item IDs: {sorted(missing)}"

IMAGE_DIR = Path("/content/vlm_v2_selected_images") / str(sample["sample_id"])
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

outfit_image_paths = []
for index, item_id in enumerate(item_ids):
    path = IMAGE_DIR / f"outfit_{index:02d}_{item_id}.jpg"
    images_by_id[item_id].save(path, format="JPEG", quality=95)
    outfit_image_paths.append(path)

recommendation_image_refs = {}
for row in recommendation_result.items:
    path = IMAGE_DIR / f"rec_{row.rank}_{row.item_id}.jpg"
    images_by_id[row.item_id].save(path, format="JPEG", quality=95)
    recommendation_image_refs[row.item_id] = path

print("Fetched outfit images:", len(outfit_image_paths))
print("Fetched recommendation images:", len(recommendation_image_refs))


## 7. Optional visual sanity check

Nhìn nhanh chính 7 ảnh mà Qwen sắp nhận.


In [ ]:
import matplotlib.pyplot as plt

display_rows = []
for index, item_id in enumerate(item_ids):
    display_rows.append((f"OUTFIT {index}\n{item_id}", images_by_id[item_id]))
for row in recommendation_result.items:
    display_rows.append((f"REC #{row.rank}\n{row.item_id}", images_by_id[row.item_id]))

fig, axes = plt.subplots(1, len(display_rows), figsize=(3 * len(display_rows), 4))
for ax, (title, image) in zip(axes, display_rows):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 8. Load Qwen **một lần** — T4 memory-safe

Canonical V2 config vẫn là 262,144 px/image. Riêng NB10 trên T4:
- cap model placement ở khoảng **10 GiB GPU** để chừa VRAM cho activations;
- reuse cùng `backend` nếu cell bị chạy lại;
- không load Qwen lần thứ hai trong cùng runtime.

Nếu người deploy có GPU VRAM lớn hơn, họ có thể dùng canonical backend/config mà không cần cap này.


In [ ]:
import gc
import torch
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

from src.vlm.config_v2 import load_vlm_config_v2
from src.vlm.qwen_backend_v2 import Qwen3VLBackendV2
from src.vlm.pipeline_v2 import VLMExplanationPipelineV2

VLM_CONFIG_PATH = REPO_ROOT / "configs" / "vlm_qwen3_vl_4b_instruct_v2.json"
vlm_config = load_vlm_config_v2(VLM_CONFIG_PATH)

gc.collect()
torch.cuda.empty_cache()

if "backend" not in globals():
    model_id = vlm_config["model"]["id"]
    print("Loading Qwen once with T4 headroom:", model_id)

    model = Qwen3VLForConditionalGeneration.from_pretrained(
        model_id,
        dtype=torch.float16,
        device_map="auto",
        max_memory={0: "10GiB", "cpu": "10GiB"},
        low_cpu_mem_usage=True,
    )
    model.eval()
    processor = AutoProcessor.from_pretrained(model_id)

    backend = Qwen3VLBackendV2(
        model=model,
        processor=processor,
        torch_module=torch,
        vision_config=vlm_config["vision"],
        model_id=model_id,
    )
else:
    print("Reusing existing Qwen backend; NOT loading a second model.")

vlm_pipeline = VLMExplanationPipelineV2(backend, vlm_config)

free, total = torch.cuda.mem_get_info()
print("GPU allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GiB")
print("GPU free     :", round(free / 1024**3, 2), "GiB")
print("GPU total    :", round(total / 1024**3, 2), "GiB")


## 9. Real Qwen inference với automatic T4 OOM fallback

Thử 65,536 px/image trước. Nếu T4 vẫn OOM, **catch lỗi bên trong cell**, dọn temporary tensors và retry cùng model ở 32,768 px/image. Không reload Qwen.

Đây chỉ là test memory profile. `used_pixels` sẽ được in ra để không nhầm với canonical deploy config.


In [ ]:
import gc
import torch

TEST_PIXEL_BUDGETS = (65536, 32768)
vlm_run = None
used_pixels = None

for pixels in TEST_PIXEL_BUDGETS:
    gc.collect()
    torch.cuda.empty_cache()

    vlm_pipeline.config["vision"]["min_pixels"] = pixels
    vlm_pipeline.config["vision"]["max_pixels"] = pixels

    free_before, _ = torch.cuda.mem_get_info()
    print(
        f"\nTrying Qwen with {pixels} px/image | "
        f"GPU free before inference: {free_before / 1024**3:.2f} GiB"
    )

    try:
        vlm_run = vlm_pipeline.explain(
            evidence,
            outfit_image_paths,
            recommendation_image_refs,
        )
        used_pixels = pixels
        break
    except torch.OutOfMemoryError as error:
        print(f"T4 OOM at {pixels} px/image: {error}")
        gc.collect()
        torch.cuda.empty_cache()
        continue

if vlm_run is None:
    raise RuntimeError(
        "Qwen vẫn OOM ở 32768 px/image. Restart runtime và chạy NB10 từ đầu; "
        "không load Qwen bằng cell khác trước notebook này."
    )

print("\nSUCCESS")
print("NB10 test pixel budget used:", used_pixels)
print("generation_attempts:", vlm_run["generation_attempts"])

print("\n=== VISUAL ANALYSIS ===")
print(json.dumps(vlm_run["visual_analysis"], indent=2, ensure_ascii=False))

print("\n=== HANDOFF / FINAL VI ===")
print(json.dumps(vlm_run["handoff"], indent=2, ensure_ascii=False))


## 10. Save run artifact

Lưu run thật + pixel budget test để nhóm biết output này chạy ở resolution nào.


In [ ]:
OUTPUT_DIR = Path("/content/drive/MyDrive/vlm_v2_runs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / f"{sample['sample_id']}_vlm_v2_nb10.json"

artifact = {
    "nb10_test_metadata": {
        "git_head": HEAD,
        "source_split": "valid",
        "gpu": torch.cuda.get_device_name(0),
        "test_pixel_budget_per_image": used_pixels,
        "canonical_pixel_budget_per_image": 262144,
        "note": "NB10 functional-test override; not frozen deploy config.",
    },
    "vlm_run": vlm_run,
}

with OUTPUT_PATH.open("w", encoding="utf-8") as stream:
    json.dump(artifact, stream, indent=2, ensure_ascii=False)

print("Saved:", OUTPUT_PATH)
